In [ ]:
# Download Medium Images Without Extensions
This notebook downloads Medium images whose URLs lack explicit file extensions and saves them into `assets/images`.
</VSCode.Cell>
<VSCode.Cell id="2" language="python">
# Import Required Libraries
import os
import re
import pathlib
import mimetypes
import requests

print('libraries imported')
</VSCode.Cell>
<VSCode.Cell id="3" language="python">
# Define Image URLs
image_urls = [
    'https://miro.medium.com/v2/resize:fit:1100/format:webp/0*-tzU8KkmM9qWPErS',
    'https://miro.medium.com/v2/resize:fit:828/0*d6_bxO4bnQC2CUlZ',
    'https://miro.medium.com/v2/resize:fit:828/0*AhP-okv0VVjvI1zh',
    'https://miro.medium.com/v2/resize:fit:828/0*qMLwkCrcwaz55Vh7',
]
print(f'Prepared {len(image_urls)} example URLs.')
</VSCode.Cell>
<VSCode.Cell id="4" language="python">
# Download Image from URL

def download_image(url):
    response = requests.get(url, timeout=30, stream=True, headers={'User-Agent': 'Mozilla/5.0'})
    response.raise_for_status()
    return response

print('download function ready')
</VSCode.Cell>
<VSCode.Cell id="5" language="python">
# Determine File Extension

def extension_from_url_or_response(url, response):
    url_path = pathlib.Path(url.split('?')[0]).suffix
    if url_path:
        return url_path

    content_type = response.headers.get('Content-Type', '')
    extension = mimetypes.guess_extension(content_type.split(';')[0].strip())
    if extension:
        return extension
    if 'webp' in content_type:
        return '.webp'
    return '.jpg'

print('extension detection ready')
</VSCode.Cell>
<VSCode.Cell id="6" language="python">
# Save Image to Images Folder

def save_image(response, folder, filename):
    folder_path = pathlib.Path(folder)
    folder_path.mkdir(parents=True, exist_ok=True)
    file_path = folder_path / filename
    with open(file_path, 'wb') as f:
        for chunk in response.iter_content(chunk_size=8192):
            if chunk:
                f.write(chunk)
    return file_path

print('save function ready')
</VSCode.Cell>
<VSCode.Cell id="7" language="python">
# Handle Multiple Downloads

folder = 'assets/images'
results = []
for url in image_urls:
    try:
        print(f'Downloading {url}')
        response = download_image(url)
        ext = extension_from_url_or_response(url, response)
        filename = f'medium-image-{len(results)+1}{ext}'
        saved_path = save_image(response, folder, filename)
        results.append(saved_path)
        print('Saved to', saved_path)
    except Exception as exc:
        print('Failed for', url, exc)

print('\nDownload finished. Files:')
for path in results:
    print(path)
